# 69. Decoupled BO：共通評価予算によるベンチマーク

測定回数ではなく累積評価費用を揃えて比較します。初期観測も費用に含め、同じseedの初期設計を共有します。少数seed・小予算の例は実行手順を理解するためのもので、一般的な性能優位性を示すものではありません。

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from robotorchan.benchmarks.decoupled import run_decoupled_benchmark
from robotorchan.benchmarks.decoupled_constrained import run_constrained_decoupled_benchmark
from robotorchan.benchmarks.decoupled_multifidelity import run_multifidelity_decoupled_benchmark
from robotorchan.benchmarks.decoupled_heterogeneous import run_heterogeneous_decoupled_benchmark
from robotorchan.benchmarks.decoupled_materials import run_materials_decoupled_benchmark
from robotorchan.benchmarks.decoupled_reporting import summarize_cost_curves
from robotorchan.benchmarks.multiobjective_problems import branin_currin
from robotorchan.benchmarks.constrained_problems import constrained_quadratic
from robotorchan.benchmarks.mixed_problems import mixed_process_yield

torch.set_num_threads(1)
seeds = [0, 1]

def plot_records(rows, metric):
    for (method, seed), group in pd.DataFrame(rows).groupby(["method", "seed"]):
        group = group.sort_values("cost")
        plt.step(group.cost, group[metric], where="post", label=f"{method}, seed={seed}")
    plt.xlabel("Cumulative evaluation cost")
    plt.ylabel(metric)
    plt.legend()
    plt.show()

def compare(run, budget):
    rows = [row for seed in seeds for decoupled in (False, True)
            for row in run(seed, decoupled)]
    assert all(row["cost"] <= budget for row in rows)
    return rows

## 1. 多目的最適化：Hypervolume

Branin-Currinの二目的を共通の参照点で評価します。推奨集合の真値Hypervolumeは大きいほど良く、観測済み点だけのHypervolumeとは区別します。KGは固定重みの意思決定基準であり、HVKGとは異なります。

In [ ]:
rows = compare(lambda seed, decoupled: run_decoupled_benchmark(
    branin_currin(), seed=seed, decoupled=decoupled, budget=20,
    n_initial=4, num_samples=8, policy="kg"), 20)
plot_records(rows, "hypervolume")
display(pd.DataFrame(rows).groupby(["method", "seed"]).tail(1))

## 2. 費用軸の整列とpaired bootstrap

測定費用が異なると結果が更新される時点も異なります。各費用格子ではその時点までの最後の値を使います。未来の観測を補間に使いません。seed軌跡単位で再標本化し、同一seedの差を計算します。2 seedの区間は説明用であり、統計的結論には不十分です。

In [ ]:
baseline = next(r["method"] for r in rows if r["method"].startswith("coupled"))
summary = summarize_cost_curves(rows, "hypervolume", list(range(16, 21)),
                               baseline=baseline, n_resamples=200)
df = pd.DataFrame(summary)
for method, g in df.groupby("method"):
    plt.step(g.cost, g["mean"], where="post", label=method)
    plt.fill_between(g.cost.to_numpy(), g.lower.to_numpy(), g.upper.to_numpy(),
                     step="post", alpha=.2)
plt.xlabel("Cumulative cost")
plt.ylabel("Hypervolume")
plt.legend()
plt.show()
for method, g in df[df.difference.notna()].groupby("method"):
    plt.step(g.cost, g.difference, where="post", label=method)
    plt.fill_between(g.cost.to_numpy(), g.difference_lower.to_numpy(),
                     g.difference_upper.to_numpy(), step="post", alpha=.2)
plt.axhline(0, color="gray", linestyle="--")
plt.xlabel("Cumulative cost")
plt.ylabel("Paired HV difference")
plt.legend()
plt.show()

## 3. 制約付き・Mixed最適化

実行可能性は推奨点の真の制約で判定します。実行不可能な推奨のSimple Regretは未定義なので、欠損をゼロに置換しません。連続問題とMixed問題は別々に比較します。

In [ ]:
for problem in [constrained_quadratic(), mixed_process_yield()]:
    costs = (3,) + (1,)*problem.n_constraints
    budget = 3*sum(costs)
    records = compare(lambda seed, decoupled: run_constrained_decoupled_benchmark(
        problem, output_costs=costs, infeasible_utility=-5.,
        seed=seed, decoupled=decoupled, n_initial=2, budget=budget,
        num_samples=4), budget)
    print(problem.name)
    plot_records(records, "feasible")
    display(pd.DataFrame(records).groupby(["method", "seed"]).tail(1))

## 4. Multi-Fidelity

低忠実度の利用有無と部分観測の利用有無を分けて比較します。Simple Regretは目標忠実度で評価し、小さいほど良い指標です。

In [ ]:
mf_rows = []
for mf in (False, True):
    mf_rows.extend(compare(lambda seed, decoupled: run_multifidelity_decoupled_benchmark(
        seed=seed, decoupled=decoupled, multifidelity=mf,
        n_initial=2, budget=36, num_samples=4), 36))
plot_records(mf_rows, "simple_regret")

## 5. 分類・異種出力

Brier Scoreは確率予測の二乗誤差で、小さいほど良い指標です。分類の種類によってクラス数が異なるので、異なる問題の絶対値を直接比較しません。

In [ ]:
for kind in ("binary", "multiclass", "ordinal"):
    records = compare(lambda seed, decoupled: run_heterogeneous_decoupled_benchmark(
        kind=kind, seed=seed, decoupled=decoupled, n_initial=3,
        budget=16, num_samples=4, epochs=10), 16)
    print(kind)
    plot_records(records, "brier_score")
    plot_records(records, "utility")

## 6. 材料探索と実行時間

強度・導電率・品質の固定重み効用を比較します。実行時間は測定費用とは別の指標で、ハードウェアや学習設定に依存します。

In [ ]:
materials = compare(lambda seed, decoupled: run_materials_decoupled_benchmark(
    seed=seed, decoupled=decoupled, n_initial=3, budget=36,
    num_samples=4, epochs=10), 36)
plot_records(materials, "utility")
plot_records(materials, "runtime_seconds")
display(pd.DataFrame(materials).groupby(["method", "seed"]).tail(1))

## 7. 結果の解釈

予算が同じでも観測数、推奨点、評価項目は異なります。比較では初期設計、参照点、費用モデル、seed、学習設定を固定し、問題ごとに結果を集計します。予算内で次の測定が不可能になった正常終了の軌跡は最後の値を保持できますが、途中失敗した実験を正常終了として扱ってはいけません。信頼区間は点ごとの区間であり、全費用軸にわたる同時信頼区間ではありません。